# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.

%load_ext dotenv
%dotenv 

In [3]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [5]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**", "*.parquet"),recursive=True)

# Print the file paths
for f in parquet_files:
    print(f)






# import random


# random.seed(42)
# parquet_files = random.sample(parquet_files, 60)

# dt_list = []
# for p_file in parquet_files:
#   #  _logs.info(f"Reading file: {s_file}")
#     dt = pd.read_parquet(p_file).assign(
#         source = os.path.basename(p_file),
#         ticker = os.path.basename(p_file).replace('.parquet', ''),
#         Date = lambda x: pd.to_datetime(x['Date'])
#     )
#     dt_list.append(dt)
# price = pd.concat(dt_list, axis = 0, ignore_index = True)

../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_2015/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_2014/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_2014/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_2013/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_2013/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_1980/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_1980/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_1987/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_1987/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_1989/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_1989/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_1988/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_1988/part.1.parquet
../../05_src/data/prices/BKTI/BKTI_1986/part.0.parquet
../../05_src/data/prices/BKTI/BKTI_1986/part.1.parquet
../../05_s

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [6]:
dd_px = dd.read_parquet(parquet_files)
print(dd_px.columns)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='object')


In [7]:
# Write your code below.

dd_px = dd.read_parquet(parquet_files).set_index("ticker")
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.sort_values('Date').assign(       # Sorting by 'Date' within each ticker to ensure we can get chronological order.
        Close_lag_1 = x['Close'].shift(1),         # Creating columns to store the 'Close' and 'Adj Close' value shifted by 1 row.
        Adj_Close_lag_1 = x['Adj Close'].shift(1),
        Close_returns = (x['Close'] / x['Close'].shift(1)) - 1,  # Calculate the daily returns based on 'Close' price as (today's Close / yesterday's Close) - 1.
        Adj_returns = (x['Adj Close'] / x['Adj Close'].shift(1)) - 1, # Calculate the daily returns based on 'Adj_Close' price as (today's Adj_Close / yesterday's Adj_Close) - 1
        hi_lo_range = x['High'] - x['Low'] # Calculate the difference between the highest and lowest price of the day
        )
)



/var/folders/hm/25wlw_0s6mg2ybf664mnchz40000gn/T/ipykernel_46948/3638079886.py:4: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [12]:
# Write your code below.

df_feat = dd_feat.compute() # convert the variable to pandas data frame

#add a new column and take the average 10 days of each ticker's return value.
df_feat['10_Day_Moving_Average'] = df_feat.groupby('ticker')['Close_returns'].transform(lambda x: x.rolling(10).mean())

df_feat



,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Close_returns,Adj_returns,hi_lo_range,10_Day_Moving_Average
ticker,,,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,NaN,7.153078,NaN
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,-0.082386,2.280043,NaN
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,0.089783,2.816525,NaN
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,-0.090909,2.592991,NaN
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,0.026562,1.385908,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2020-03-26,9.610000,9.940000,9.260000,9.590000,9.590000,60500.0,ZEUS.csv,2020,14.170000,12.977926,-0.031052,-0.031052,0.670000,-0.009660
ZEUS,2020-03-27,9.330000,9.330000,8.700000,8.700000,8.700000,52900.0,ZEUS.csv,2020,13.730000,12.574939,0.030590,0.030590,0.530000,-0.008381
ZEUS,2020-03-30,8.810000,9.760000,8.700000,9.680000,9.680000,73700.0,ZEUS.csv,2020,14.150000,12.959603,-0.021908,-0.021908,0.600000,-0.014199


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

Yes, because in Dask the data is split into many parts and then the rolling averages are taken from each part of the dataset. However, we want to make perform the rolling average on the whole dataset. Dask does not combine the data from the separated parts of the dataset, therefore the rolling averages could be wrong. Using pandas will allow us to perform the rolling average on the whole dataset at once.

+ Would it have been better to do it in Dask? Why?

No, since the data is not that large, pandas is able to simply complete the moving average calculation.


(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.